# Task 2 - Data Engineering / NLP Pipeline (Week 2)
**Notebook:** `notebooks/Task2.ipynb`
**Aligned lectures:** Feature Engineering at Scale; Distributed Data Wrangling; Data Partitioning & Sampling; Data Pipelines.

We clean the text, build the **sentiment label**, engineer numeric features, and assemble a **PySpark NLP `Pipeline`** (tokenise -> stopwords -> TF-IDF -> assemble -> scale).

In [1]:
# === Colab bootstrap: install Spark (run once per session) ===
!pip -q install pyspark==3.5.1
import os, time
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("7006SCN_AmazonReviews_Sentiment")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "12g")
         .config("spark.driver.maxResultSize", "2g")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 21.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
Spark version: 3.5.1
Spark UI: http://c99e3985222d:4040


In [2]:
# Mount Google Drive so data/models persist across the six notebooks.
from google.colab import drive
drive.mount('/content/drive')
BASE   = '/content/drive/MyDrive/7006SCN'
RAW    = BASE + '/raw'                      # downloaded jsonl.gz
PARQ   = BASE + '/reviews_parquet'          # raw -> parquet
PROC   = BASE + '/reviews_processed'        # after Task2 NLP pipeline
EXPORT = BASE + '/tableau'                  # Task6 exports
import os
for p in (BASE, RAW, EXPORT):
    os.makedirs(p, exist_ok=True)
print('Base:', BASE)

Mounted at /content/drive
Base: /content/drive/MyDrive/7006SCN


## 2.1 Ingestion with partition count (screenshot)

In [3]:
from pyspark.sql import functions as F
df = spark.read.parquet(PARQ)
print("Initial partitions:", df.rdd.getNumPartitions())
print("Rows:", df.count())
df = df.repartition(64)
print("After repartition:", df.rdd.getNumPartitions())

Initial partitions: 26
Rows: 20812945
After repartition: 64


## 2.2 Missing values, label creation, cleaning (justified)
The label is **sentiment**: positive if `rating>=4`, negative if `rating<=2`; rating **3 is dropped** as ambiguous/neutral (standard practice that sharpens the decision boundary). Rows with null `text`/`rating` cannot be used and are removed. We lowercase and strip non-letters to reduce noise/sparsity.

In [4]:
clean = (df
    .filter(F.col("text").isNotNull() & F.col("rating").isNotNull())
    .filter(F.col("rating") != 3)
    .withColumn("label", (F.col("rating") >= 4).cast("double"))
    .dropDuplicates(["text","user_id"])
    .drop("user_id"))                      # data minimisation (ethics)

# class balance (screenshot - motivates Task 4 imbalance handling)
clean.groupBy("label").count().show()

+-----+--------+
|label|   count|
+-----+--------+
|  1.0|14797353|
|  0.0| 3993147|
+-----+--------+



## 2.3 Feature engineering (justified)
- **`clean_text`** - lowercase, letters-only: shrinks vocabulary, removes punctuation noise.
- **`text_length`, `word_count`, `title_length`** - longer reviews often correlate with stronger sentiment.
- **`verified`** (verified_purchase) and **`helpful`** (helpful_vote) - credibility/engagement signals combined with the text vector.

In [5]:
fe = (clean
    .withColumn("clean_text", F.lower(F.regexp_replace(F.col("text"), "[^a-zA-Z ]", " ")))
    .withColumn("text_length", F.length("text"))
    .withColumn("word_count", F.size(F.split(F.col("clean_text"), "\\s+")))
    .withColumn("title_length", F.length(F.coalesce(F.col("title"), F.lit(""))))
    .withColumn("verified", F.col("verified_purchase").cast("double"))
    .withColumn("helpful", F.col("helpful_vote").cast("double")))
fe = fe.fillna({"verified":0.0, "helpful":0.0})
fe.select("clean_text","text_length","word_count","verified","helpful","label").show(5, truncate=50)

+--------------------------------------------------+-----------+----------+--------+-------+-----+
|                                        clean_text|text_length|word_count|verified|helpful|label|
+--------------------------------------------------+-----------+----------+--------+-------+-----+
|don t you just love it when a product is well d...|        305|        53|     1.0|    0.0|  1.0|
|this is the third purchase of this muffin pan i...|        311|        61|     1.0|    1.0|  1.0|
|i ve had this for about     months and it is gr...|        137|        32|     1.0|    8.0|  1.0|
|                                           bad buy|          7|         2|     1.0|    0.0|  0.0|
|complaints about the shipment  received the cas...|       1036|       186|     1.0|  104.0|  1.0|
+--------------------------------------------------+-----------+----------+--------+-------+-----+
only showing top 5 rows



## 2.4 Why a PySpark NLP `Pipeline`?
A `Pipeline` fixes the exact text-vectorisation recipe, **fit on training data only**, so the same vocabulary/IDF weights apply to test data and future reviews - no leakage, fully reproducible and serialisable. Stages in order:

1. `RegexTokenizer` - split clean text into word tokens (min length 2).
2. `StopWordsRemover` - drop uninformative words ("the", "and").
3. `CountVectorizer` - build a fixed vocabulary (top 20,000 terms, minDF=5) and term-frequency vectors. (CountVectorizer is chosen over HashingTF so we can map model weights back to **words** for explainability in Task 5.)
4. `IDF` - down-weight common terms -> TF-IDF.
5. `VectorAssembler` - concatenate TF-IDF with the numeric features.
6. `StandardScaler` (withMean=False) - unit variance while keeping values **non-negative** (required by Naive Bayes in Task 3).

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (RegexTokenizer, StopWordsRemover, CountVectorizer,
                                IDF, VectorAssembler, StandardScaler)

tok = RegexTokenizer(inputCol="clean_text", outputCol="tokens", pattern="\\s+", minTokenLength=2)
swr = StopWordsRemover(inputCol="tokens", outputCol="filtered")
cv  = CountVectorizer(inputCol="filtered", outputCol="tf", vocabSize=20000, minDF=5.0)
idf = IDF(inputCol="tf", outputCol="tfidf")
asm = VectorAssembler(inputCols=["tfidf","text_length","word_count","title_length","verified","helpful"],
                      outputCol="features_raw", handleInvalid="keep")
scl = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True)
pipeline = Pipeline(stages=[tok, swr, cv, idf, asm, scl])

## 2.5 Split, fit pipeline on train only, transform, persist

In [7]:
train, test = fe.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)

keep = ["features","label"]
train_t = model.transform(train).select(*keep)
test_t  = model.transform(test ).select(*keep)
train_t.write.mode("overwrite").parquet(PROC + "_train")
test_t.write.mode("overwrite").parquet(PROC + "_test")
model.write().overwrite().save(BASE + "/fitted_pipeline")

# Save vocabulary for Task 5 explainability (map weights -> words)
import json
vocab = model.stages[2].vocabulary
with open(EXPORT + "/vocab.json","w") as f: json.dump(vocab, f)
print("Train:", train_t.count(), "Test:", test_t.count(), "Vocab:", len(vocab))

Train: 15030817 Test: 3759683 Vocab: 20000


## Version control
>=3 commits, e.g. `Task2: label + text cleaning`, `Task2: numeric feature engineering`, `Task2: TF-IDF NLP pipeline (CountVectorizer->IDF->scaler)`.